# 1. 环境配置

## 1.1 python 环境准备

In [2]:
! pip install openai==2.11.0 gradio==6.2.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1 numexpr==2.14.1 wikipedia==1.4.0

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/90/7f/340847023184305a6378d75ec71e1dd38a942dfe71b7c29314b8fbe26948/arxiv-2.3.1-py3-none-any.whl (11 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/1f/67/ffe750b5452eb66de788c34e7d21ec6d886abb4d7c43ad1dc88ceb3d998f/numexpr-2.14.1-cp312-cp312-win_amd64.whl (160 kB)
  Using cached wikipedia-1.4.0-py3-none-any.whl
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/4e/eb/c96d64137e29ae17d83ad2552470bafe3a7a915e85434d9942077d7fd011/feedparser-6.0.12-py3-none-any.whl (81 kB)
  Using cached sgmllib3k-1.0.0-py3-none-any.whl

   ---------------- ----------------------- 2/5 [feedparser]
   ---------------------------------------- 5/5 [arxiv]



## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [3]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. LangChain V1.0 ReAct Agent 搭建

## 2.1 简介

看完了如何通过基础的 python 代码手搓 ReAct Agent，下面我们来看看如何使用 LangChain 框架来复现该系统。

对于搭建一个基于 LangChain 的 ReAct Agent 系统，我们还是需要以下几个零部件：

- LLM 模型（大脑）
- Memory（记忆）
- 工具（行动能力）
- 提示词模版（思维方式）

只不过这里的零部件我们使用更快速和简便的方式即可实现。LangChain 框架帮助我们将很多的内容写好，我们只需要考虑具体的工具设置以及运行逻辑即可。

在最新的 LangChain V1.0 版本中，LangChain 官方已经全面将 Agent 组件中的相关内容接入到 LangGraph 中。LangGraph 是 LangChain 团队在 2024 年推出的新一代框架，用于构建具备“状态、记忆与多步推理能力”的智能体（Agent）。

它以“图（Graph）”为核心思想：把每个模型调用、工具执行、判断逻辑都视为一个节点，通过节点间的有向边来定义信息流动，从而让复杂的智能体流程（如 ReAct、对话管理、多智能体协作、人工中断等）都能用图结构清晰地表达。

相比旧版 LangChain 的线性链式调用，LangGraph 提供了可控、可追踪、可持久化的执行流，支持状态保存、检查点（Checkpoint）、时间回溯（Time Travel）等功能，是目前官方推荐的 Agent 开发核心框架。

## 2.2 LLM 模型

由于在 LangGraph 中，需要大模型支持工具调用（Function Calling）。因为通义千问明确在 LangChain 文档中表示支持，因此后续我们将使用 Qwen 系列模型进行演示。

In [4]:
from langchain_community.chat_models import ChatTongyi
import os

llm = ChatTongyi(
  api_key=os.environ.get("DASHSCOPE_API_KEY"), 
  model="qwen-max")

response = llm.invoke("你好，请介绍一下你自己")

print(response.content)

你好！我是Qwen，是阿里云开发的一款超大规模语言模型。我被设计用来帮助人们回答问题、创作文字，比如写故事、写公文、写邮件、写剧本等等，还能表达观点，玩游戏等。我的目标是成为人类的得力助手，帮助人们提高工作效率和生活质量。有什么我可以帮到你的吗？


## 2.3 Memory 记忆

在新版本里，我们不再需要通过 RunnableWithMessageHistory 的方式进行记忆的保留，在 LangGraph 下我们使用 InMemorySaver() 的方式进行保留。

所谓的短期记忆（Short-term Memory），实际上指的是系统仅在当前对话轮次中保存的临时上下文信息，用于维持一次连续的对话逻辑或局部推理过程。

在 LangGraph 中，InMemorySaver() 的作用就是在智能体（Agent）或图（Graph）执行的过程中，临时保存模型的输入输出、对话内容、工具调用记录以及节点状态等信息，从而使系统在一次会话（session）中能够“记住”先前的交互历史。

In [5]:
from langgraph.checkpoint.memory import InMemorySaver 
memory = InMemorySaver()

## 2.4 System Prompt 系统提示词

在过去，我们都是要通过提示词模版来指挥 Agent

但是这种方式有两个问题：
- 易出错（模板拼错就崩）
- 不通用（每种模型格式不同）

所以在 LangChain V0.3 开始就重构了 Agent 层，从而让开发者只描述能力，而不是再造 prompt。

所以只需要输入模型、工具列表、记忆以及系统提示词（会在每一次传入给模型时才传入，不会放入记忆中）

In [6]:
system_prompt = "You are a helpful assistant"

## 2.5 Tool 工具

### 2.5.1 内置工具

根据LangChain的官方文档，目前LangChain提供支持的有以下几类工具：
- 搜索工具：用于在线搜索，包括 Bing、Google、DuckDuckGo 等搜索接口。返回内容一般包括：URL、标题、摘要等。
- 代码辅助工具：支持 Python、JavaScript 等语言的代码执行环境，可用于复杂计算或文件处理。
- 生产力工具：用于对接 Gmail、Office365、Slack、Jira 等办公/协作平台，实现任务自动化。
- 网页浏览：用于浏览网页、交互操作、信息抓取（如Hyperbrowser 等）。
- 数据库：支持与数据库交互，包括 SQL、Spark SQL 等数据库的读取与操作。
- 其他常用工具：维基百科、ArXiv、Python REPL

假如我们需要使用 LangChain 内置的工具，我们首先需要使用一个 load_tools 工具，然后在里面写入对应工具的名称：

In [7]:
from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["arxiv"])

假如我们想添加更多的工具，我们可以在后面不断添加（部分依赖语言模型来执行计算或生成代码需要传入一个 LLM 实例）：

In [8]:
tools = load_tools(["arxiv","llm-math", "wikipedia"], llm=llm)

### 2.3.2 自定义工具

但是在我们当前的 Agent 设定下，我们并不需要使用上 LangChain 里内置的工具。

我们所使用的是我们自己定义的 calculate 工具和 average_dog_weight 工具。

因此下面我们就来看看如何将这些函数转化为 LangChain 中可使用的工具：

In [9]:
def calculate(what):
    return eval(what)

print(calculate("3 + 7 * 2"))   # 返回 17
print(calculate("10 / 4"))      # 返回 2.5

def average_dog_weight(name):
  if name in "Scottish Terrier": 
    return("Scottish Terriers average 20 lbs")
  elif name in "Border Collie":
    return("a Border Collies average weight is 37 lbs")
  elif name in "Toy Poodle":
    return("a toy poodles average weight is 7 lbs")
  else:
    return("An average dog weights 50 lbs")

print(average_dog_weight("Scottish Terrier")) 
# 返回 "Scottish Terriers average 20 lbs"
print(average_dog_weight("Labrador"))     
# 返回 "An average dog weights 50 lbs"

17
2.5
Scottish Terriers average 20 lbs
An average dog weights 50 lbs


在 LangChain里，定义工具的主要方法为使用 @tool 装饰器。这种方式最为简洁，只需在普通函数上方加上 @tool 装饰器,并且在内部加上文档字符串作为工具的介绍以及参数的说明，就能自动将该函数注册为一个可供智能体调用的工具。

系统会自动提取函数的名称、参数说明和文档字符串（上一个提示词里对工具的示例）作为工具的描述（写入提示词中），非常适合快速创建一些简单的功能型工具。例如：计算表达式、查询天气等。

In [10]:
from langchain.tools import tool

@tool
def calculate(what: str) -> str:
    """
    calculate:
    e.g. calculate: 4 * 7 / 3
    Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
    """
    return str(eval(what))

假如我们不想在函数内部写入文档字符串，我们也可以在 @tool 中添加一个属性 description 去写入同样的内容。

这里的 name 表示工具的名称（假如没有就默认使用函数名称）。description 代表的函数的介绍，也就是与前面文档字符串的作用一致。

In [11]:
from langchain.tools import tool

@tool(description="""
  calculate:
  e.g. calculate: 4 * 7 / 3
  Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
  """)
def calculate(what: str) -> str:
    return str(eval(what))

在 LangChain v1.0 版本中还新增了一个基于 Pydantic 的输入格式审查工具。

我们可以先定义一个 Pydantic 输入模型，通过 what: str 明确输入的类型是字符串，然后介绍需要输入里面的内容是什么。比如这里介绍的就是输入的是一个字符串格式的数学表达式。

然后再装饰器里我们需要将这个类载入到 @tool 的 args_schema 参数中，然后后续在模型调用工具的时候就能获取这部分信息。

In [12]:
from pydantic import BaseModel, Field

class CalcInput(BaseModel):
    """Input for math calculation"""
    what: str = Field(description="A mathematical expression, e.g., '4 * 7 / 3'")
    
from langchain.tools import tool

@tool(args_schema=CalcInput)
def calculate(what: str) -> str:
    """
    calculate:
    e.g. calculate: 4 * 7 / 3
    Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
    """
    return str(eval(what))

在定义好两个工具后，我们可以将两个工具组合起来形成工具列表，等待后续传给 ReAct Agent 进行使用：

In [13]:
from langchain.tools import tool

@tool
def calculate(what: str) -> str:
  """
  calculate:
  e.g. calculate: 4 * 7 / 3
  Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary
  """
  return str(eval(what))

@tool
def average_dog_weight(name: str) -> str:
  """
  average_dog_weight:
  e.g. average_dog_weight: Collie
  returns average weight of a dog when given the breed
  """
  name = name.lower()
  if "scottish terrier" in name:
    return "Scottish Terriers average 20 lbs"
  elif "border collie" in name:
    return "A Border Collie's average weight is 37 lbs"
  elif "toy poodle" in name:
    return "A Toy Poodle's average weight is 7 lbs"
  else:
    return "An average dog weighs 50 lbs"
  
tools = [calculate, average_dog_weight]

## 2.6 系统组装

在准备好了一些基础的组件以后，我们使用 create_agent 的方法将这些内容组合起来：

In [14]:
from langchain.agents import create_agent

agent = create_agent(model=llm, 
           tools=tools, 
           system_prompt=system_prompt, 
           checkpointer=memory)

然后我们同样需要设置 thread_id 并将问题进行传入：

In [15]:
result1 = agent.invoke({"messages": [{"role": "user", "content": "I have 2 dogs, a border collie and a scottish terrier. What is their combined weight? Could you double it?"}]}, config={"configurable": {"thread_id": "user_1"}})
print(result1)

{'messages': [HumanMessage(content='I have 2 dogs, a border collie and a scottish terrier. What is their combined weight? Could you double it?', additional_kwargs={}, response_metadata={}, id='c893b456-ff83-45be-bac4-a1e356e85ce4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"name": "border collie"}', 'name': 'average_dog_weight'}, 'id': 'call_794c5aabfa924fe092916e', 'index': 0, 'type': 'function'}, {'function': {'arguments': '{"name": "scottish terrier"}', 'name': 'average_dog_weight'}, 'id': 'call_290ffd207a024e1eba15fa', 'index': 1, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '2a794018-dabb-4397-b09d-683cbf9c6448', 'token_usage': {'input_tokens': 369, 'output_tokens': 47, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 416}}, id='lc_run--019b9af7-6836-7a41-8cda-e556749874e2-0', tool_calls=[{'name': 'average_dog_weight', 'args': {'name': 'border collie'}, 'id

假如只需要答案的话：

In [16]:
print(result1["messages"][-1].content)

The calculation confirms that the combined weight of the two dogs, when doubled, is 114 lbs.


## 2.7 使用 ChatInterface 实现页面构建
假如想快速构建一个 Gradio 的对话页面来测试我们的 agent，可以使用以下方式来实现：

- 定义一个 agent :

In [17]:
from langchain.agents import create_agent
agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt, checkpointer=memory)

- 基于 agent 定义一个 agent_response 函数：

In [18]:
def agent_response(content, history):
  result1 = agent.invoke({"messages": [{"role": "user", "content": content}]}, config={"configurable": {"thread_id": "user_1"}})
  return result1["messages"][-1].content

- 创建并发布 ChatInterface 页面：

In [20]:
import gradio as gr
demo = gr.ChatInterface(fn=agent_response)
demo.launch()

c:\Users\76391\.conda\envs\langchain_2026_1_7\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


这样就可以去测试回复的内容是否准确，但假如希望能够看到工具调用情况，需要使用流式输出。

# 3. 总结

通过对比框架开发和非框架两种方式，我们可以看到：
- LangChain 等框架化的 ReAct 实现，通过高度抽象的组件化设计，将工具调用、提示词构建、推理循环和记忆管理整合到统一的执行流程中。
- 它让开发者能够专注于业务逻辑本身，而无需手动处理底层的 Action 解析、状态传递和多轮对话维护。这种方式更符合现代智能体开发的趋势——模块化、可扩展、可复用、可持久。
- 而非框架的手写 ReAct 实现，则以最直接的形式揭示了智能体推理的“底层机制”。但这种方式对开发者要求更高，难以维护和扩展。
> 框架式实现 → 适用于工程化、教学系统或生产部署，体现了智能体的可扩展与可持续特征。

> 非框架式实现 → 适用于教学演示或研究实验，帮助我们深入理解 ReAct 的核心原理与思维闭环。